# Phase 2: Intermediate Python (Modular & Pythonic Engineering)
Covers Comprehensions, Functional Built-ins, Iterators & Generators, OOP, Dataclasses, Dynamic Introspection, Exception Handling, Context Managers, File I/O, Serialization, and Standard Library Modules.

## 1. Comprehensions & Functional Built-ins (`map`, `filter`, `functools`)

In [ ]:
from functools import reduce, lru_cache, partial

# List, Dict, and Set Comprehensions
raw_tokens = ["  AI ", "llm", "PyTorch  ", "llm", "RAG"]
clean_tokens = [t.strip().lower() for t in raw_tokens]
unique_tokens = {t for t in clean_tokens}
token_lengths = {t: len(t) for t in unique_tokens}

print("Cleaned:", clean_tokens)
print("Token Lengths:", token_lengths)

# map, filter, and reduce
confidences = [0.45, 0.88, 0.92, 0.31, 0.79]
high_conf = list(filter(lambda x: x >= 0.75, confidences))
scaled_conf = list(map(lambda x: round(x * 100, 1), high_conf))
total_score = reduce(lambda acc, val: acc + val, high_conf, 0.0)

print("Filtered & Scaled (%):", scaled_conf)
print("Total Score:", round(total_score, 2))

# functools.lru_cache (Caching expensive calls) & partial
@lru_cache(maxsize=128)
def compute_embedding_mock(text: str) -> tuple:
    return tuple(ord(c) / 255.0 for c in text[:4])

print("Cached Embedding:", compute_embedding_mock("agent"))

## 2. Iterators, Generators (`iter`, `next`, `yield`) & `itertools`

In [ ]:
import itertools

# Generator for memory-efficient batch streaming in ML
def batch_generator(dataset: list, batch_size: int):
    for i in range(0, len(dataset), batch_size):
        yield dataset[i : i + batch_size]

sample_data = list(range(1, 11))
stream = batch_generator(sample_data, batch_size=3)

print("First batch via next():", next(stream))
for batch in stream:
    print("Remaining batch:", batch)

# itertools essentials (chain, islice, product for grid search)
learning_rates = [1e-3, 1e-4]
batch_sizes = [16, 32]
hyperparam_grid = list(itertools.product(learning_rates, batch_sizes))
print("Hyperparameter Grid:", hyperparam_grid)

## 3. OOP, Dunder Methods, Introspection & `dataclasses`

In [ ]:
from dataclasses import dataclass, field, asdict

# Dataclasses for clean ML/AI training configurations
@dataclass
class TrainingConfig:
    model_name: str
    epochs: int = 10
    lr: float = 3e-4
    tags: list[str] = field(default_factory=lambda: ["baseline"])

config = TrainingConfig(model_name="Qwen-2.5")
print("Config as Dict:", asdict(config))

# OOP, Inheritance, @property, @classmethod, @staticmethod, and Dunders
class BaseLayer:
    def __init__(self, name: str):
        self.name = name

class DenseLayer(BaseLayer):
    def __init__(self, name: str, units: int):
        super().__init__(name)
        self._units = units
        self.weights = [0.1] * units

    @property
    def units(self) -> int:
        return self._units

    @classmethod
    def from_config(cls, cfg: dict):
        return cls(name=cfg["name"], units=cfg["units"])

    @staticmethod
    def relu(x: float) -> float:
        return max(0.0, x)

    def __len__(self) -> int:
        return len(self.weights)

    def __getitem__(self, idx: int) -> float:
        return self.weights[idx]

    def __call__(self, inputs: list[float]) -> float:
        return self.relu(sum(w * x for w, x in zip(self.weights, inputs)))

    def __repr__(self) -> str:
        return f"DenseLayer(name={self.name!r}, units={self._units})"

layer = DenseLayer.from_config({"name": "fc1", "units": 4})
print(repr(layer), "| Output:", layer([1.0, 2.0, -1.0, 0.5]))

# Dynamic Introspection Built-ins
print("Is callable?:", callable(layer))
print("Is subclass?:", issubclass(DenseLayer, BaseLayer))
print("Has weights?:", hasattr(layer, "weights"))
setattr(layer, "trainable", True)
print("Get trainable:", getattr(layer, "trainable"))

## 4. Exception Handling, Debugging & Context Managers (`with`)

In [ ]:
from contextlib import contextmanager
import time

class ModelInferenceError(Exception):
    pass

@contextmanager
def inference_timer(task_name: str):
    start = time.perf_counter()
    try:
        yield
    finally:
        elapsed = (time.perf_counter() - start) * 1000
        print(f"[{task_name}] completed in {elapsed:.2f} ms")

with inference_timer("Token Generation"):
    try:
        tokens = ["Hello", "world"]
        if not tokens:
            raise ModelInferenceError("Empty token stream")
    except ModelInferenceError as e:
        print("Inference failed:", e)
    else:
        print("Generated:", " ".join(tokens))

## 5. Files, Paths, Formats (`pathlib`, `json`, `csv`, `tomllib`, `tempfile`, `zipfile`, `sqlite3`, `shelve`)

In [ ]:
import csv
import json
import sqlite3
import tempfile
import tomllib
import zipfile
from pathlib import Path

# Using tempfile and pathlib for safe local file operations
with tempfile.TemporaryDirectory() as tmp_dir:
    base_path = Path(tmp_dir)

    # 1. JSON Read/Write
    json_path = base_path / "metrics.json"
    json_path.write_text(json.dumps({"accuracy": 0.94, "loss": 0.12}), encoding="utf-8")
    loaded_json = json.loads(json_path.read_text(encoding="utf-8"))
    print("Loaded JSON:", loaded_json)

    # 2. CSV Read/Write
    csv_path = base_path / "dataset.csv"
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["id", "prompt", "label"])
        writer.writeheader()
        writer.writerow({"id": 1, "prompt": "Explain RAG", "label": "tech"})

    # 3. Zipfile archive creation
    zip_path = base_path / "bundle.zip"
    with zipfile.ZipFile(zip_path, "w") as zf:
        zf.write(json_path, arcname="metrics.json")
        zf.write(csv_path, arcname="dataset.csv")
    print("Archive contents:", zipfile.ZipFile(zip_path).namelist())

# 4. TOML parsing (Python 3.11+ built-in tomllib)
toml_string = '[model]\nname = "Llama-3"\nquantization = "int4"'
print("Parsed TOML:", tomllib.loads(toml_string))

# 5. SQLite3 In-Memory Database (Used for local Agent memory / prompt caching)
with sqlite3.connect(":memory:") as conn:
    conn.execute("CREATE TABLE logs (id INTEGER PRIMARY KEY, prompt TEXT, tokens INT)")
    conn.execute("INSERT INTO logs (prompt, tokens) VALUES (?, ?)", ("Hello AI", 12))
    row = conn.execute("SELECT * FROM logs").fetchone()
    print("SQLite Row:", row)

## 6. Standard Library Essentials (`collections`, `re`, `hashlib`, `random`, `math`, `statistics`, `datetime`, `zoneinfo`, `argparse`, `logging`)

In [ ]:
import argparse
import hashlib
import logging
import math
import random
import re
import statistics
from collections import Counter, defaultdict, deque
from datetime import datetime
from zoneinfo import ZoneInfo

# 1. collections (Counter for token frequencies, deque for sliding window memory)
token_counts = Counter(["ai", "agent", "ai", "rag", "ai", "agent"])
memory_window = deque(maxlen=3)
for msg in ["msg1", "msg2", "msg3", "msg4"]:
    memory_window.append(msg)
print("Top Tokens:", token_counts.most_common(2), "| Sliding Window:", list(memory_window))

# 2. Regular Expressions (re) & Hashlib (Document deduplication in RAG)
raw_doc = "Contact support@example.com or admin@ai.org for API keys."
redacted_doc = re.sub(r"[\w.-]+@[\w.-]+", "[EMAIL_REDACTED]", raw_doc)
doc_hash = hashlib.sha256(redacted_doc.encode("utf-8")).hexdigest()[:16]
print("Redacted:", redacted_doc, "| Hash:", doc_hash)

# 3. Math, Statistics & Random Seeding
random.seed(42)
latencies = [random.uniform(10.0, 50.0) for _ in range(5)]
print("Mean Latency:", round(statistics.mean(latencies), 2), "| Cosine(0):", math.cos(0))

# 4. Timezone-aware timestamps
now_pkt = datetime.now(ZoneInfo("Asia/Karachi"))
print("Current Timestamp (PKT):", now_pkt.isoformat())

# 5. Argparse CLI simulation & Logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
parser = argparse.ArgumentParser(description="Train AI Model")
parser.add_argument("--epochs", type=int, default=5)
parser.add_argument("--lr", type=float, default=0.001)
args = parser.parse_args(["--epochs", "20", "--lr", "0.0005"])
logging.info(f"Parsed CLI Args -> Epochs: {args.epochs}, LR: {args.lr}")